# Source-Grounded Business Summaries · real-time and batch example

**Available on [AWS Marketplace](https://aws.amazon.com/marketplace/pp/prodview-fpr4ku72wwntg), version 0.1.0.** AWS confirms Public product visibility and a Released offer. The release package completed actual AWS validation against the exact image; the full seller real-time, batch and cleanup rehearsal passed. Subscribe in your account before deploying. No model output is fabricated or saved in this notebook.

Subscribe, choose **Continue to configuration**, select `us-east-1`, and copy the model-package ARN into the `GROUNDED_SUMMARY_MODEL_PACKAGE_ARN` environment variable. No seller package ARN is supplied as a default. The notebook checks the completed Marketplace container and exact qualified image digest; your active subscription supplies access. A buyer-account subscription transaction has not been rehearsed by the seller.

**Default Run All is offline.** Review the notebook, then change `RUN_DEMO = False` to `True` to enable AWS setup, real-time deployment, invocation and cleanup. Batch is separately disabled until you set `RUN_BATCH = True`. Keep `RUN_DEMO` enabled while cleaning up.

Run in your own SageMaker notebook role session in `us-east-1`, with `boto3` installed. The notebook infers that execution role and creates a uniquely named private encrypted evidence bucket. Outside a role session, set `SAGEMAKER_EXECUTION_ROLE_ARN`; optionally use an existing private bucket with `SAGEMAKER_DEMO_BUCKET`. Existing buckets are checked and never reconfigured. The execution role must trust SageMaker and allow the purchased package plus the private input/output prefix. Your caller needs permission to create, describe, invoke and delete the demo SageMaker resources, pass that role, read the role identity, and create/configure the fresh S3 evidence bucket. If bucket setup fails, inspect the printed bucket name in S3. Evidence storage remains billable after compute cleanup. Request capacity for one `ml.g6e.4xlarge` endpoint and, optionally, one `ml.g6e.2xlarge` Batch Transform instance.

**AWS infrastructure charges are separate from software charges and any software trial.** This example creates one GPU endpoint until you delete it; batch compute also incurs charges while running. Execute the cleanup cells, even after a failure. All example text is fictional. Review every model result: literal source evidence does not establish semantic support.

In [ ]:
import json
import os
import re
import time
import uuid

import boto3
from botocore.exceptions import ClientError, WaiterError

# Opt in only after reviewing costs, permissions and cleanup. Default Run All is offline.
RUN_DEMO = False
RUN_BATCH = False
APPROVED_IMAGE_DIGEST = "sha256:6d38d9384206b0b77181db4f47167dde5bd23fe9e4c7ff86075dfcc2b468c917"
REGION = "us-east-1"


def validate_package_arn(arn):
    if not re.fullmatch(r"arn:aws:sagemaker:us-east-1:\d{12}:model-package/[A-Za-z0-9-]+", arn):
        raise ValueError("Set GROUNDED_SUMMARY_MODEL_PACKAGE_ARN to your subscribed us-east-1 package ARN.")


def validate_package(package):
    if package.get("ModelPackageStatus") != "Completed":
        raise RuntimeError("The subscribed package is not Completed.")
    containers = package.get("InferenceSpecification", {}).get("Containers", [])
    if len(containers) != 1 or not containers[0].get("ProductId"):
        raise RuntimeError("Expected one AWS Marketplace container; verify your subscription and package ARN.")
    image_digest = containers[0].get("ImageDigest") or containers[0].get("Image", "").partition("@")[2]
    if image_digest != APPROVED_IMAGE_DIGEST:
        raise RuntimeError("Package image differs from the release-approved digest.")


configured_arn = os.environ.get("GROUNDED_SUMMARY_MODEL_PACKAGE_ARN", "").strip()
if RUN_DEMO:
    # Validate configuration before constructing any client or creating resources.
    if not re.fullmatch(r"sha256:[0-9a-f]{64}", APPROVED_IMAGE_DIGEST):
        raise RuntimeError("Unreleased candidate: no certified image digest is approved.")
    validate_package_arn(configured_arn)
    if any(
        globals().get(key, False)
        for key in (
            "model_created",
            "config_created",
            "endpoint_created",
            "transform_created",
        )
    ):
        raise RuntimeError("Clean up the existing demo before rerunning setup; its resource names are preserved.")
    session = boto3.Session(region_name=REGION)
    sm = session.client("sagemaker")
    validate_package(sm.describe_model_package(ModelPackageName=configured_arn))
    runtime = session.client("sagemaker-runtime")
    s3 = session.client("s3")
    identity = session.client("sts").get_caller_identity()
    account_id = identity["Account"]

if RUN_DEMO:
    run_name = "ss-summary-demo-" + time.strftime("%Y%m%d-%H%M%S", time.gmtime()) + "-" + uuid.uuid4().hex[:6]
    prefix = f"grounded-summary-demo/{run_name}"


def prepare_demo_access(session, identity, region):
    """Use the notebook role and a fresh private bucket unless explicitly configured."""
    account = identity["Account"]
    role = os.environ.get("SAGEMAKER_EXECUTION_ROLE_ARN", "").strip()
    if not role:
        assumed = identity["Arn"].partition(":assumed-role/")[2]
        if "/" not in assumed:
            raise RuntimeError("Run in a SageMaker role session or configure SAGEMAKER_EXECUTION_ROLE_ARN.")
        role_name = assumed.rsplit("/", 1)[0].rsplit("/", 1)[-1]
        role = session.client("iam").get_role(RoleName=role_name)["Role"]["Arn"]
    if not role.startswith(f"arn:aws:iam::{account}:role/"):
        raise RuntimeError("The SageMaker execution role must belong to your AWS account.")
    storage = session.client("s3")
    bucket_name = os.environ.get("SAGEMAKER_DEMO_BUCKET", "").strip()
    if bucket_name:
        storage.head_bucket(Bucket=bucket_name, ExpectedBucketOwner=account)
        blocks = storage.get_public_access_block(
            Bucket=bucket_name,
            ExpectedBucketOwner=account,
        )["PublicAccessBlockConfiguration"]
        keys = ("BlockPublicAcls", "IgnorePublicAcls", "BlockPublicPolicy", "RestrictPublicBuckets")
        if not all(blocks.get(k) is True for k in keys):
            raise RuntimeError("The configured evidence bucket must block public access; it was not modified.")
    else:
        bucket_name = f"sagemaker-summary-demo-{account}-{region}-{uuid.uuid4().hex[:8]}"
        params = {"Bucket": bucket_name}
        if region != "us-east-1":
            params["CreateBucketConfiguration"] = {"LocationConstraint": region}
        storage.create_bucket(**params)
        print(f"Created private evidence bucket {bucket_name}; it is retained with the demo inputs/outputs.")
        # Only alter the fresh uniquely named bucket after successful creation.
        storage.put_public_access_block(
            Bucket=bucket_name,
            ExpectedBucketOwner=account,
            PublicAccessBlockConfiguration={
                "BlockPublicAcls": True,
                "IgnorePublicAcls": True,
                "BlockPublicPolicy": True,
                "RestrictPublicBuckets": True,
            },
        )
        storage.put_bucket_encryption(
            Bucket=bucket_name,
            ExpectedBucketOwner=account,
            ServerSideEncryptionConfiguration={
                "Rules": [{"ApplyServerSideEncryptionByDefault": {"SSEAlgorithm": "AES256"}}],
            },
        )
    return role, bucket_name


if RUN_DEMO:
    role_arn, bucket = prepare_demo_access(session, identity, REGION)
    tags = [{"Key": "SoftwareSushiDemoRun", "Value": run_name}]
    model_created = False
    config_created = False
    endpoint_created = False
    transform_created = False

## Bounded multilingual requests

Use one structured request per invocation, or one request per line in a line-split batch. Limits: 1–64 unique segments; 2,000 characters per segment; 16,000 combined ID-plus-text characters; 96 KiB body; 3,072 actual input tokens; 1,024 output tokens. Oversized inputs are rejected without truncation. Owner is an explicitly named person only, and deadline is an explicit valid ISO date only; otherwise null. This service accepts text, not audio, images, or PDFs.

In [ ]:
examples = {
    "en": "Maya Chen will send the signed transition plan by 2027-06-18.",
    "es": "Lucía Mena enviará el cronograma firmado antes del 2027-06-18.",
    "pt": "João Luís enviará o cronograma assinado até 2027-06-18.",
}
requests = [{"language": language, "segments": [{"id": "s1", "text": text}]} for language, text in examples.items()]

## Create one temporary real-time endpoint

Run this cell only when ready to test. Wait for `InService`, then invoke immediately and delete the endpoint using the next cleanup cell. If creation or waiting fails, run cleanup before retrying. Keep this notebook open until cleanup finishes.

In [ ]:
if RUN_DEMO:
    sm.create_model(
        ModelName=run_name,
        ExecutionRoleArn=role_arn,
        PrimaryContainer={"ModelPackageName": configured_arn},
        EnableNetworkIsolation=True,
        Tags=tags,
    )
    model_created = True
    sm.create_endpoint_config(
        EndpointConfigName=run_name,
        ProductionVariants=[
            {
                "VariantName": "AllTraffic",
                "ModelName": run_name,
                "InstanceType": "ml.g6e.4xlarge",
                "InitialInstanceCount": 1,
                "InitialVariantWeight": 1,
                "ContainerStartupHealthCheckTimeoutInSeconds": 600,
                "ModelDataDownloadTimeoutInSeconds": 1200,
            }
        ],
        Tags=tags,
    )
    config_created = True
    sm.create_endpoint(EndpointName=run_name, EndpointConfigName=run_name, Tags=tags)
    endpoint_created = True
    sm.get_waiter("endpoint_in_service").wait(
        EndpointName=run_name,
        WaiterConfig={"Delay": 30, "MaxAttempts": 80},
    )

In [ ]:
if RUN_DEMO:
    for request in requests:
        response = runtime.invoke_endpoint(
            EndpointName=run_name,
            ContentType="application/json",
            Accept="application/json",
            Body=json.dumps(request, ensure_ascii=False).encode("utf-8"),
        )
        try:
            result = json.loads(response["Body"].read())
        finally:
            response["Body"].close()
        assert result["model"] == "grounded-summary" and result["version"] == "0.1.0"
        assert result["review_required"] is True
        source = {s["id"]: s["text"] for s in request["segments"]}
        for item in result["summary"] + result["actions"]:
            evidence = item["evidence"]
            text = source[evidence["segment_id"]]
            assert evidence["quote"] == text and evidence["start"] == 0 and evidence["end"] == len(text)
        print(json.dumps(result, ensure_ascii=False, indent=2))
    # Exact citation location is checked above; semantic support still requires review.

In [ ]:
if RUN_DEMO:
    # Stop endpoint charges before the optional batch example. Retain the model for batch.
    if endpoint_created:
        sm.delete_endpoint(EndpointName=run_name)
        sm.get_waiter("endpoint_deleted").wait(
            EndpointName=run_name,
            WaiterConfig={"Delay": 15, "MaxAttempts": 80},
        )
        endpoint_created = False
    if config_created:
        sm.delete_endpoint_config(EndpointConfigName=run_name)
        config_created = False

## Optional line-split Batch Transform

Set `RUN_BATCH = True` before setup to enable this cell. It needs the separate GPU batch quota. It uploads three fictional requests to your private bucket using server-side encryption. Batch compute stops when the job completes. Input/output objects are retained in your bucket for inspection; remove only this run's prefix later if appropriate.

In [ ]:
if RUN_DEMO and RUN_BATCH:
    input_key = f"{prefix}/input/examples.jsonl"
    s3.put_object(
        Bucket=bucket,
        Key=input_key,
        ExpectedBucketOwner=account_id,
        Body=("\n".join(json.dumps(r, ensure_ascii=False) for r in requests) + "\n").encode("utf-8"),
        ContentType="application/jsonlines",
        ServerSideEncryption="AES256",
    )
    sm.create_transform_job(
        TransformJobName=run_name,
        ModelName=run_name,
        BatchStrategy="SingleRecord",
        MaxConcurrentTransforms=1,
        MaxPayloadInMB=1,
        TransformInput={
            "DataSource": {"S3DataSource": {"S3DataType": "S3Prefix", "S3Uri": f"s3://{bucket}/{input_key}"}},
            "ContentType": "application/jsonlines",
            "SplitType": "Line",
        },
        TransformOutput={
            "S3OutputPath": f"s3://{bucket}/{prefix}/output/",
            "Accept": "application/jsonlines",
            "AssembleWith": "Line",
        },
        TransformResources={"InstanceType": "ml.g6e.2xlarge", "InstanceCount": 1},
        Tags=tags,
    )
    transform_created = True
    sm.get_waiter("transform_job_completed_or_stopped").wait(
        TransformJobName=run_name,
        WaiterConfig={"Delay": 30, "MaxAttempts": 120},
    )
    job = sm.describe_transform_job(TransformJobName=run_name)
    if job["TransformJobStatus"] != "Completed":
        raise RuntimeError("Batch did not complete; inspect the job and run cleanup.")
    output = s3.get_object(
        Bucket=bucket,
        Key=f"{prefix}/output/examples.jsonl.out",
        ExpectedBucketOwner=account_id,
    )
    try:
        for line in output["Body"].read().decode("utf-8").splitlines():
            print(json.dumps(json.loads(line), ensure_ascii=False, indent=2))
    finally:
        output["Body"].close()

## Cleanup after success or failure

Run this cell even if an earlier operation failed. It acts only on this unique demo run. It stops an unfinished batch job, deletes any remaining endpoint/configuration, and removes the demo model after compute stops. It preserves S3 input/output evidence. Check the AWS console if a cleanup call fails; closing the notebook does not stop an endpoint.

In [ ]:
if RUN_DEMO:
    cleanup_errors = []
    batch_active = False
    if transform_created:
        try:
            job = sm.describe_transform_job(TransformJobName=run_name)
            status = job["TransformJobStatus"]
            batch_active = status in ("InProgress", "Stopping")
            if status == "InProgress":
                sm.stop_transform_job(TransformJobName=run_name)
            if batch_active:
                sm.get_waiter("transform_job_completed_or_stopped").wait(
                    TransformJobName=run_name,
                    WaiterConfig={"Delay": 15, "MaxAttempts": 80},
                )
                batch_active = False
            transform_created = False
        except (ClientError, WaiterError) as exc:
            cleanup_errors.append(exc)
    # Endpoint cleanup still runs if a batch operation failed.
    if endpoint_created:
        try:
            state = sm.describe_endpoint(EndpointName=run_name)["EndpointStatus"]
            if state != "Deleting":
                sm.delete_endpoint(EndpointName=run_name)
            sm.get_waiter("endpoint_deleted").wait(
                EndpointName=run_name,
                WaiterConfig={"Delay": 15, "MaxAttempts": 80},
            )
            endpoint_created = False
        except (ClientError, WaiterError) as exc:
            cleanup_errors.append(exc)
    if config_created and not endpoint_created:
        try:
            sm.delete_endpoint_config(EndpointConfigName=run_name)
            config_created = False
        except ClientError as exc:
            cleanup_errors.append(exc)
    if model_created and not endpoint_created and not batch_active:
        try:
            sm.delete_model(ModelName=run_name)
            model_created = False
        except ClientError as exc:
            cleanup_errors.append(exc)
    if cleanup_errors:
        raise RuntimeError(
            "Some cleanup calls failed; inspect this demo run in AWS before closing."
        ) from cleanup_errors[0]
    print(f"Demo compute removed. Private evidence remains at s3://{bucket}/{prefix}/")